In [1]:
!pip install pymupdf
!pip install pymupdf pytesseract pdf2image
!apt-get update
!apt-get install -y poppler-utils tesseract-ocr
!pip install pymupdf pytesseract pdf2image

!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 54.6 MB/s eta 0:00:00:00:0100:01
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Ign:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Ign:5 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,978 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,908 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]  
Get:12 http://security.u

In [2]:
import fitz 
import pytesseract
from pdf2image import convert_from_path
import PIL.Image
import os
from docx import Document 

def universal_cv_reader(file_path):
    text = ""
    try:
        if not os.path.exists(file_path):
            return f"Error: File not found at {file_path}"

        file_ext = file_path.lower().split('.')[-1]

        if file_ext == 'pdf':
            doc = fitz.open(file_path)
            for page in doc:
                text += page.get_text()
            doc.close()

            if len(text.strip()) < 50:
                images = convert_from_path(file_path)
                for img in images:
                    text += pytesseract.image_to_string(img)

        elif file_ext == 'docx':
            doc = Document(file_path)
            text = " ".join([para.text for para in doc.paragraphs])

        elif file_ext in ['jpg', 'jpeg', 'png', 'webp']:
            text = pytesseract.image_to_string(PIL.Image.open(file_path))

    except Exception as e:
        return f" Error: {str(e)}"
    
    return " ".join(text.split())

In [4]:
import re

def basic_text_processor(text):
    # --- Case Folding ---
    text = text.lower()

    # --- Noise Removal (Cleaning) ---
    text = re.sub(r'\S*@\S*\s?', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # --- Normalization ---
    text = " ".join(text.split())

    # ---  Word Tokenization ---
    tokens = text.split()
    
    return " ".join(tokens)

In [3]:
def clean_category_name(name):
    clean_name = re.sub(r'\s*resumes?\s*', '', name, flags=re.IGNORECASE)
    clean_name = re.sub(r'\s*datasets?\s*', '', clean_name, flags=re.IGNORECASE)
    return clean_name.strip().capitalize()

In [4]:
import os
import pandas as pd
import re

input_base_paths = [
    '/kaggle/input/datasets/snehaanbhawal/resume-dataset/data/data',
    '/kaggle/input/datasets/youssefkhalil/resumes-images-datasets/Resumes Datasets/Bing_images',
    '/kaggle/input/datasets/youssefkhalil/resumes-images-datasets/Resumes Datasets/Scrapped_Resumes', 
    '/kaggle/input/datasets/youssefkhalil/resumes-images-datasets/Resumes Datasets/resume_database',
    '/kaggle/input/datasets/hadikp/resume-data-pdf/Resumes PDF',
    '/kaggle/input/datasets/palaksood97/resume-dataset/Resumes'
]


all_file_tasks = []
source_counts = {}

for base_path in input_base_paths:
    if not os.path.exists(base_path):
        print(f" path not found: {base_path}")
        continue
    
    source_name = base_path.split('/')[-1] if base_path.split('/')[-1] != 'data' else base_path.split('/')[-2]
    source_counts[source_name] = 0
    
    is_uncategorized = "palaksood97/resume-dataset/Resumes" in base_path
    
    for root, dirs, files in os.walk(base_path):
        raw_category = os.path.basename(root)
        category = "Unknown" if is_uncategorized else clean_category_name(raw_category)
        
        for file in files:
            if file.lower().endswith(('.pdf', '.docx', '.jpg', '.png', '.webp')):
                source_counts[source_name] += 1
                all_file_tasks.append({
                    'path': os.path.join(root, file),
                    'filename': file,
                    'category': category,
                    'source': source_name
                })

print("\n" + "="*50)
print(f"{'Dataset Source':<35} | {'num of CVs':<10}")
print("-"*50)
for source, count in source_counts.items():
    print(f"{source:<35} | {count:<10}")
print("-"*50)
print(f"{'Total':<35} | {len(all_file_tasks):<10}")
print("="*50 + "\n")


Dataset Source                      | num of CVs
--------------------------------------------------
data                                | 2484      
Bing_images                         | 2802      
Scrapped_Resumes                    | 4436      
resume_database                     | 5402      
Resumes PDF                         | 8905      
Resumes                             | 228       
--------------------------------------------------
Total                               | 24257     



In [ ]:
import os
import pandas as pd
import re
from tqdm import tqdm

#Part 1

START_INDEX = 0       
BATCH_SIZE = 5000    
END_INDEX = START_INDEX + BATCH_SIZE

current_batch = all_file_tasks[START_INDEX:END_INDEX]

print(f"current_batch from {START_INDEX} to {min(END_INDEX, len(all_file_tasks))}...")

data_list = []

for task in tqdm(current_batch):
    try:
        raw_text = universal_cv_reader(task['path'])
        
        clean_text = basic_text_processor(raw_text)
        
        if clean_text.strip():
            data_list.append({
                'filename': task['filename'],
                'category': task['category'], 
                'text': clean_text
            })
    except Exception as e:
        continue

if data_list:
    df = pd.DataFrame(data_list)
    output_file = f'/kaggle/working/resumes_batch_{START_INDEX}_{END_INDEX}.csv'
    df.to_csv(output_file, index=False)

In [ ]:
import os
import pandas as pd
import re
from tqdm import tqdm

#Part 2

START_INDEX = 5001       
BATCH_SIZE = 10000     
END_INDEX = START_INDEX + BATCH_SIZE

current_batch = all_file_tasks[START_INDEX:END_INDEX]

print(f"current_batch from {START_INDEX} to {min(END_INDEX, len(all_file_tasks))}...")

data_list = []

for task in tqdm(current_batch):
    try:
        raw_text = universal_cv_reader(task['path'])
        
        clean_text = basic_text_processor(raw_text)
        
        if clean_text.strip():
            data_list.append({
                'filename': task['filename'],
                'category': task['category'], 
                'text': clean_text
            })
    except Exception as e:
        continue

if data_list:
    df = pd.DataFrame(data_list)
    output_file = f'/kaggle/working/resumes_batch_{START_INDEX}_{END_INDEX}.csv'
    df.to_csv(output_file, index=False)

In [ ]:
import os
import pandas as pd
import re
from tqdm import tqdm

#Part 3

START_INDEX = 10001       
BATCH_SIZE = 15000     
END_INDEX = START_INDEX + BATCH_SIZE

current_batch = all_file_tasks[START_INDEX:END_INDEX]

print(f"current_batch from {START_INDEX} to {min(END_INDEX, len(all_file_tasks))}...")

data_list = []

for task in tqdm(current_batch):
    try:
        raw_text = universal_cv_reader(task['path'])
        
        clean_text = basic_text_processor(raw_text)
        
        if clean_text.strip():
            data_list.append({
                'filename': task['filename'],
                'category': task['category'], 
                'text': clean_text
            })
    except Exception as e:
        continue

if data_list:
    df = pd.DataFrame(data_list)
    output_file = f'/kaggle/working/resumes_batch_{START_INDEX}_{END_INDEX}.csv'
    df.to_csv(output_file, index=False)

In [ ]:
import os
import pandas as pd
import re
from tqdm import tqdm

#Part 4

START_INDEX = 15001       
BATCH_SIZE = 20000     
END_INDEX = START_INDEX + BATCH_SIZE

current_batch = all_file_tasks[START_INDEX:END_INDEX]

print(f"current_batch from {START_INDEX} to {min(END_INDEX, len(all_file_tasks))}...")

data_list = []

for task in tqdm(current_batch):
    try:
        raw_text = universal_cv_reader(task['path'])
        
        clean_text = basic_text_processor(raw_text)
        
        if clean_text.strip():
            data_list.append({
                'filename': task['filename'],
                'category': task['category'], 
                'text': clean_text
            })
    except Exception as e:
        continue

if data_list:
    df = pd.DataFrame(data_list)
    output_file = f'/kaggle/working/resumes_batch_{START_INDEX}_{END_INDEX}.csv'
    df.to_csv(output_file, index=False)

In [ ]:
import os
import pandas as pd
import re
from tqdm import tqdm

#Part 5

START_INDEX = 20001       
BATCH_SIZE = 24256     
END_INDEX = START_INDEX + BATCH_SIZE

current_batch = all_file_tasks[START_INDEX:END_INDEX]

print(f"current_batch from {START_INDEX} to {min(END_INDEX, len(all_file_tasks))}...")

data_list = []

for task in tqdm(current_batch):
    try:
        raw_text = universal_cv_reader(task['path'])
        
        clean_text = basic_text_processor(raw_text)
        
        if clean_text.strip():
            data_list.append({
                'filename': task['filename'],
                'category': task['category'], 
                'text': clean_text
            })
    except Exception as e:
        continue

if data_list:
    df = pd.DataFrame(data_list)
    output_file = f'/kaggle/working/resumes_batch_{START_INDEX}_{END_INDEX}.csv'
    df.to_csv(output_file, index=False)

In [ ]:
import pandas as pd
import glob
import os

file_pattern = '/kaggle/input/datasets/ibrahemmohamed18/cvs-dataset/resumes_batch_*.csv'
all_batch_files = glob.glob(file_pattern)

print(f" find {len(all_batch_files)} files.")

li = []
for filename in all_batch_files:
    df_temp = pd.read_csv(filename, index_col=None, header=0)
    li.append(df_temp)

final_df = pd.concat(li, axis=0, ignore_index=True)

output_master = '/kaggle/working/final_24k_resumes_master.csv'
final_df.to_csv(output_master, index=False)